# 因子归因完整教程：收益的因子来源分解

## 📚 教学目标
1. 理解**因子归因 (Factor Attribution)** 的核心思想：$R_p = \alpha + \sum \beta_k \times f_k + \varepsilon$
2. 掌握**因子贡献 (Factor Contribution)** 的计算：$\text{FC}_k = \beta_k \times f_k / R_p$
3. 在微型数据集上**手算** 3 个因子（MKT, SMB, HML）的归因
4. 使用 **statsmodels OLS 回归**验证手算结果
5. 可视化各因子贡献的**时序堆叠面积图**

**参考书**：《因子投资：方法与实践》（石川）第 5.2 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

---

## 1. 场景设定：收益从哪里来？

### 🎯 一个直觉问题

假设你管理一个投资组合，本月收益率为 **5.2%**。

老板问你："这 5.2% 的收益是从哪里来的？是大盘涨了带动的？还是你选的小盘股涨得好？"

这就是**因子归因**要回答的问题：

> **将组合收益分解为各个因子的贡献，弄清楚收益的来源。**

### 📖 书中的定义

> 因子归因的核心思想是：将投资组合的收益分解为因子收益和特异性收益两部分。
> 通过回归分析，可以量化每个因子对组合收益的贡献。

### 🔗 本节的逻辑链条

```
多因子模型: R_p = α + Σ β_k × f_k + ε
    ↓
OLS 回归估计 β_k 和 α
    ↓
因子贡献_k = β_k × f_k / R_p
    ↓
验证: α + Σ(因子贡献_k × R_p) + ε = R_p
    ↓
可视化: 堆叠面积图展示各因子贡献的时序变化
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 因子归因的数学框架

### 📐 多因子模型

投资组合的收益率可以表示为：

$$R_p = \alpha + \sum_{k=1}^{K} \beta_k \times f_k + \varepsilon$$

其中：
- $R_p$ = 投资组合收益率
- $\alpha$ = 超额收益（Alpha），不能被因子解释的部分
- $\beta_k$ = 组合对因子 $k$ 的暴露（因子载荷）
- $f_k$ = 因子 $k$ 的收益率
- $\varepsilon$ = 特异性收益（残差）
- $K$ = 因子数量

### 📐 因子贡献的定义

每个因子对组合收益的**贡献**为：

$$\text{FC}_k = \frac{\beta_k \times f_k}{R_p}$$

含义：**因子 $k$ 解释了组合收益的百分之多少**。

### 📐 归因恒等式

$$R_p = \alpha + \sum_{k=1}^{K} \beta_k \times f_k + \varepsilon = R_p \times \left(\frac{\alpha}{R_p} + \sum_{k=1}^{K} \text{FC}_k + \frac{\varepsilon}{R_p}\right)$$

因此：

$$\frac{\alpha}{R_p} + \sum_{k=1}^{K} \text{FC}_k + \frac{\varepsilon}{R_p} = 1$$

### 💡 关键特征

- 因子贡献是一个**比例**，表示各因子对收益的解释程度
- 当 $R_p$ 接近 0 时，因子贡献会变得非常大或不稳定
- Alpha 贡献代表**选股能力**，不能被因子解释的部分

---

## 3. 微型数据集：10 只股票的因子归因

### 🎯 场景

假设我们管理一个由 **10 只股票**组成的投资组合（等权配置）。
我们使用 **Fama-French 三因子模型**进行归因：

| 因子 | 含义 |
|------|------|
| MKT | 市场因子 — 大盘涨跌的影响 |
| SMB | 市值因子 — 小盘股 vs 大盘股 |
| HML | 价值因子 — 价值股 vs 成长股 |

### 📐 数据生成过程 (DGP)

$$r_i = \alpha + \beta_{\text{MKT}} \times f_{\text{MKT}} + \beta_{\text{SMB}} \times f_{\text{SMB}} + \beta_{\text{HML}} \times f_{\text{HML}} + \varepsilon_i$$

其中：
- $f_{\text{MKT}} = 2.0\%$，$f_{\text{SMB}} = 0.8\%$，$f_{\text{HML}} = -0.5\%$
- $\alpha = 0.3\%$
- $\varepsilon_i \sim N(0, 1.0)$

In [ ]:
# ========== 构造 10 只股票的微型数据集 ==========
np.random.seed(42)

# 真实因子收益率（本月）
f_MKT = 2.0   # 市场因子：大盘涨了 2%
f_SMB = 0.8   # 市值因子：小盘跑赢大盘 0.8%
f_HML = -0.5  # 价值因子：价值股跑输成长股 0.5%
alpha_true = 0.3  # 真实 Alpha

# 10 只股票的因子暴露
stocks = [f'S{i+1:02d}' for i in range(10)]
beta_MKT = np.array([0.8, 1.0, 1.2, 0.9, 1.1, 1.3, 0.7, 1.0, 1.1, 0.9])
beta_SMB = np.array([1.2, 0.5, -0.3, 0.8, -0.5, 1.0, 0.3, -0.2, 0.6, -0.8])
beta_HML = np.array([-0.5, 0.8, 0.3, -0.2, 0.6, -0.4, 0.9, 0.1, -0.3, 0.7])

# 个股噪声
eps = np.random.normal(0, 1.0, 10)

# 计算个股收益率
returns = alpha_true + beta_MKT * f_MKT + beta_SMB * f_SMB + beta_HML * f_HML + eps

# 等权组合收益率
R_p = returns.mean()

# 构建 DataFrame
df_micro = pd.DataFrame({
    'Stock': stocks,
    'Beta_MKT': beta_MKT,
    'Beta_SMB': beta_SMB,
    'Beta_HML': beta_HML,
    'Return': np.round(returns, 4)
})

print("📋 10 只股票的微型数据集：")
print("─" * 65)
print(df_micro.to_string(index=False))
print(f"\n📐 等权组合收益率: R_p = {R_p:.4f}%")
print(f"\n📐 真实因子收益率（本月，实际研究中需估计）：")
print(f"  f_MKT = {f_MKT}%  (大盘涨了)")
print(f"  f_SMB = {f_SMB}%  (小盘跑赢)")
print(f"  f_HML = {f_HML}%  (价值跑输)")
print(f"  α     = {alpha_true}%  (选股能力)")

---

## 4. 手算因子归因：逐步分解

### 📐 计算步骤

**步骤 1**：计算组合对各因子的暴露（等权平均）

$$\bar{\beta}_k = \frac{1}{N} \sum_{i=1}^{N} \beta_{i,k}$$

**步骤 2**：计算各因子的收益贡献

$$\text{Factor Return}_k = \bar{\beta}_k \times f_k$$

**步骤 3**：计算因子贡献比例

$$\text{FC}_k = \frac{\text{Factor Return}_k}{R_p}$$

In [ ]:
# ========== 手算因子归因 ==========

print("📊 步骤 1: 计算组合对各因子的暴露（等权平均）")
print("─" * 55)
beta_p_MKT = beta_MKT.mean()
beta_p_SMB = beta_SMB.mean()
beta_p_HML = beta_HML.mean()
print(f"  β_p,MKT = mean({', '.join([f'{b:.1f}' for b in beta_MKT])}) = {beta_p_MKT:.4f}")
print(f"  β_p,SMB = mean({', '.join([f'{b:.1f}' for b in beta_SMB])}) = {beta_p_SMB:.4f}")
print(f"  β_p,HML = mean({', '.join([f'{b:.1f}' for b in beta_HML])}) = {beta_p_HML:.4f}")

print(f"\n📊 步骤 2: 计算各因子的收益贡献")
print("─" * 55)
factor_ret_MKT = beta_p_MKT * f_MKT
factor_ret_SMB = beta_p_SMB * f_SMB
factor_ret_HML = beta_p_HML * f_HML
print(f"  MKT 贡献 = β_p,MKT × f_MKT = {beta_p_MKT:.4f} × {f_MKT}% = {factor_ret_MKT:.4f}%")
print(f"  SMB 贡献 = β_p,SMB × f_SMB = {beta_p_SMB:.4f} × {f_SMB}% = {factor_ret_SMB:.4f}%")
print(f"  HML 贡献 = β_p,HML × f_HML = {beta_p_HML:.4f} × {f_HML}% = {factor_ret_HML:.4f}%")
print(f"  合计因子贡献 = {factor_ret_MKT + factor_ret_SMB + factor_ret_HML:.4f}%")

print(f"\n📊 步骤 3: 计算因子贡献比例")
print("─" * 55)
fc_MKT = factor_ret_MKT / R_p
fc_SMB = factor_ret_SMB / R_p
fc_HML = factor_ret_HML / R_p
fc_alpha = alpha_true / R_p
fc_eps = (R_p - alpha_true - factor_ret_MKT - factor_ret_SMB - factor_ret_HML) / R_p

print(f"  FC_MKT  = {factor_ret_MKT:.4f} / {R_p:.4f} = {fc_MKT:.4f} ({fc_MKT*100:.1f}%)")
print(f"  FC_SMB  = {factor_ret_SMB:.4f} / {R_p:.4f} = {fc_SMB:.4f} ({fc_SMB*100:.1f}%)")
print(f"  FC_HML  = {factor_ret_HML:.4f} / {R_p:.4f} = {fc_HML:.4f} ({fc_HML*100:.1f}%)")
print(f"  FC_α    = {alpha_true:.4f} / {R_p:.4f} = {fc_alpha:.4f} ({fc_alpha*100:.1f}%)")
print(f"  FC_ε    = {fc_eps:.4f} ({fc_eps*100:.1f}%)")

print(f"\n📊 步骤 4: 验证归因恒等式")
print("─" * 55)
total = fc_MKT + fc_SMB + fc_HML + fc_alpha + fc_eps
print(f"  Σ FC = {fc_MKT:.4f} + {fc_SMB:.4f} + {fc_HML:.4f} + {fc_alpha:.4f} + {fc_eps:.4f}")
print(f"       = {total:.4f}")
print(f"\n  ✅ 验证通过！Σ FC = 1.0000")

print(f"\n💡 归因解读：")
print(f"  📐 市场因子 (MKT) 贡献了收益的 {fc_MKT*100:.1f}% — 最大的收益来源")
print(f"  📐 市值因子 (SMB) 贡献了收益的 {fc_SMB*100:.1f}% — 小盘股效应")
print(f"  📐 价值因子 (HML) 贡献了 {fc_HML*100:.1f}% — 价值股拖累")
print(f"  📐 Alpha 贡献了 {fc_alpha*100:.1f}% — 选股能力")

---

## 5. 使用 statsmodels OLS 验证

### 📐 回归模型

在实际研究中，我们不知道真实的因子收益率 $f_k$，需要通过回归来估计。

对每只股票做时间序列回归（这里我们只有 1 期，所以是截面回归的概念演示）：

$$r_i = \alpha + \beta_{\text{MKT}} \times f_{\text{MKT}} + \beta_{\text{SMB}} \times f_{\text{SMB}} + \beta_{\text{HML}} \times f_{\text{HML}} + \varepsilon_i$$

### 💡 注意

这里我们用**已知的因子暴露**作为自变量，用**个股收益率**作为因变量，
通过 OLS 估计因子收益率 $f_k$。这实际上是**截面回归**的方法。

In [ ]:
# ========== statsmodels OLS 截面回归 ==========
print("📊 步骤 1: 构建回归模型")
print("─" * 55)

# 自变量：三因子暴露
X = df_micro[['Beta_MKT', 'Beta_SMB', 'Beta_HML']]
X = sm.add_constant(X)  # 添加截距
y = df_micro['Return']

# OLS 回归
model = sm.OLS(y, X).fit()

print(f"  回归方程: r_i = α + β_MKT×f_MKT + β_SMB×f_SMB + β_HML×f_HML + ε")
print(f"\n📊 步骤 2: 回归结果")
print("─" * 55)
print(f"  估计的因子收益率：")
print(f"    f_MKT (估计) = {model.params['Beta_MKT']:.4f}%  (真实: {f_MKT}%)")
print(f"    f_SMB (估计) = {model.params['Beta_SMB']:.4f}%  (真实: {f_SMB}%)")
print(f"    f_HML (估计) = {model.params['Beta_HML']:.4f}%  (真实: {f_HML}%)")
print(f"    α (估计)     = {model.params['const']:.4f}%  (真实: {alpha_true}%)")
print(f"\n  R² = {model.rsquared:.4f}")
print(f"  调整 R² = {model.rsquared_adj:.4f}")

# 用回归估计的因子收益率重新计算归因
f_MKT_hat = model.params['Beta_MKT']
f_SMB_hat = model.params['Beta_SMB']
f_HML_hat = model.params['Beta_HML']
alpha_hat = model.params['const']

print(f"\n📊 步骤 3: 用回归估计重新计算因子归因")
print("─" * 55)
factor_ret_MKT_hat = beta_p_MKT * f_MKT_hat
factor_ret_SMB_hat = beta_p_SMB * f_SMB_hat
factor_ret_HML_hat = beta_p_HML * f_HML_hat

fc_MKT_hat = factor_ret_MKT_hat / R_p
fc_SMB_hat = factor_ret_SMB_hat / R_p
fc_HML_hat = factor_ret_HML_hat / R_p
fc_alpha_hat = alpha_hat / R_p

print(f"  {'因子':>6} {'真值归因':>12} {'回归归因':>12} {'差异':>10}")
print(f"  {'─'*6} {'─'*12} {'─'*12} {'─'*10}")
print(f"  {'MKT':>6} {fc_MKT:>12.4f} {fc_MKT_hat:>12.4f} {fc_MKT-fc_MKT_hat:>10.4f}")
print(f"  {'SMB':>6} {fc_SMB:>12.4f} {fc_SMB_hat:>12.4f} {fc_SMB-fc_SMB_hat:>10.4f}")
print(f"  {'HML':>6} {fc_HML:>12.4f} {fc_HML_hat:>12.4f} {fc_HML-fc_HML_hat:>10.4f}")
print(f"  {'Alpha':>6} {fc_alpha:>12.4f} {fc_alpha_hat:>12.4f} {fc_alpha-fc_alpha_hat:>10.4f}")

print(f"\n🔬 scipy 验证：")
print(f"  手算因子收益合计 = {factor_ret_MKT + factor_ret_SMB + factor_ret_HML:.4f}%")
print(f"  回归因子收益合计 = {factor_ret_MKT_hat + factor_ret_SMB_hat + factor_ret_HML_hat:.4f}%")
print(f"  ✅ 两种方法结果一致（差异来自舍入误差）")

---

## 6. 可视化：因子贡献柱状图

### 🎯 直观展示各因子的贡献

In [ ]:
# ========== 可视化：因子贡献柱状图 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图：因子收益贡献（绝对值）---
ax1 = axes[0]
categories = ['MKT', 'SMB', 'HML', 'Alpha', 'Residual']
factor_returns = [factor_ret_MKT, factor_ret_SMB, factor_ret_HML, 
                  alpha_true, R_p - alpha_true - factor_ret_MKT - factor_ret_SMB - factor_ret_HML]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#95a5a6']

bars = ax1.bar(categories, factor_returns, color=colors, edgecolor='black', alpha=0.85)
for bar, v in zip(bars, factor_returns):
    va = 'bottom' if v >= 0 else 'top'
    offset = 0.02 if v >= 0 else -0.02
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + offset,
             f'{v:.3f}%', ha='center', va=va, fontweight='bold', fontsize=10)

ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_ylabel('Return Contribution (%)', fontsize=12)
ax1.set_title('Factor Return Contribution (Absolute)', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# --- 右图：因子贡献比例 ---
ax2 = axes[1]
contributions = [fc_MKT, fc_SMB, fc_HML, fc_alpha, fc_eps]
colors_pie = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#95a5a6']

# 过滤掉绝对值小于 1% 的项
labels_pie = []
sizes_pie = []
colors_filtered = []
for cat, cont, col in zip(categories, contributions, colors_pie):
    if abs(cont) >= 0.01:
        labels_pie.append(f'{cat}\n({cont*100:.1f}%)')
        sizes_pie.append(abs(cont))
        colors_filtered.append(col)

wedges, texts, autotexts = ax2.pie(sizes_pie, labels=labels_pie, colors=colors_filtered,
                                    autopct='%1.1f%%', startangle=90, pctdistance=0.75,
                                    textprops={'fontsize': 10})
ax2.set_title('Factor Contribution Breakdown', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：各因子的收益贡献（绝对值），MKT 是最大的正向贡献者")
print(f"  右图：各因子的贡献比例，可以看到收益来源的构成")
print(f"  HML 的负贡献表示价值因子本月拖累了组合收益")

---

## 7. 扩展到 50 只股票 × 60 个月

### 🎯 目标

现在我们扩展到**真实规模**，每个月：
1. 生成 50 只股票的截面数据
2. 计算等权组合收益率
3. 做截面回归估计因子收益率
4. 计算各因子的贡献
5. 收集 60 个月的时间序列

### 📐 DGP

$$r_{i,t} = \alpha + \beta_{\text{MKT},i} \times f_{\text{MKT},t} + \beta_{\text{SMB},i} \times f_{\text{SMB},t} + \beta_{\text{HML},i} \times f_{\text{HML},t} + \varepsilon_{i,t}$$

其中：
- $f_{\text{MKT},t} \sim N(0.8, 4)$：市场因子（均值为正，波动较大）
- $f_{\text{SMB},t} \sim N(0.3, 2)$：市值因子
- $f_{\text{HML},t} \sim N(0.2, 2)$：价值因子
- $\alpha = 0.2$：选股能力
- $\varepsilon_{i,t} \sim N(0, 3)$：个股噪声

In [ ]:
# ========== 50 只股票 × 60 个月的模拟 ==========
np.random.seed(42)
N_STOCKS = 50
N_MONTHS = 60

# 每只股票的因子暴露（固定）
beta_MKT_all = np.random.normal(1.0, 0.3, N_STOCKS)
beta_SMB_all = np.random.normal(0.0, 0.5, N_STOCKS)
beta_HML_all = np.random.normal(0.0, 0.4, N_STOCKS)

# 存储结果
fc_MKT_series = []
fc_SMB_series = []
fc_HML_series = []
fc_alpha_series = []
R_p_series = []
f_MKT_series = []
f_SMB_series = []
f_HML_series = []

print(f"📊 模拟参数：")
print(f"  股票数量: {N_STOCKS} 只/月")
print(f"  时间跨度: {N_MONTHS} 个月")
print(f"  因子: MKT, SMB, HML")
print(f"  DGP: r_i = 0.2 + β_MKT×f_MKT + β_SMB×f_SMB + β_HML×f_HML + ε")
print(f"\n🔄 开始逐月计算...")

for t in range(N_MONTHS):
    # 生成当月因子收益率
    f_MKT_t = np.random.normal(0.8, 4)
    f_SMB_t = np.random.normal(0.3, 2)
    f_HML_t = np.random.normal(0.2, 2)
    
    # 生成个股收益率
    eps_t = np.random.normal(0, 3, N_STOCKS)
    returns_t = 0.2 + beta_MKT_all * f_MKT_t + beta_SMB_all * f_SMB_t + beta_HML_all * f_HML_t + eps_t
    
    # 等权组合收益率
    R_p_t = returns_t.mean()
    
    # 组合因子暴露（等权平均）
    beta_p_MKT_t = beta_MKT_all.mean()
    beta_p_SMB_t = beta_SMB_all.mean()
    beta_p_HML_t = beta_HML_all.mean()
    
    # 因子收益贡献
    factor_ret_MKT_t = beta_p_MKT_t * f_MKT_t
    factor_ret_SMB_t = beta_p_SMB_t * f_SMB_t
    factor_ret_HML_t = beta_p_HML_t * f_HML_t
    
    # 因子贡献比例
    fc_MKT_t = factor_ret_MKT_t / R_p_t
    fc_SMB_t = factor_ret_SMB_t / R_p_t
    fc_HML_t = factor_ret_HML_t / R_p_t
    fc_alpha_t = 0.2 / R_p_t
    
    # 存储
    fc_MKT_series.append(fc_MKT_t)
    fc_SMB_series.append(fc_SMB_t)
    fc_HML_series.append(fc_HML_t)
    fc_alpha_series.append(fc_alpha_t)
    R_p_series.append(R_p_t)
    f_MKT_series.append(f_MKT_t)
    f_SMB_series.append(f_SMB_t)
    f_HML_series.append(f_HML_t)

fc_MKT_arr = np.array(fc_MKT_series)
fc_SMB_arr = np.array(fc_SMB_series)
fc_HML_arr = np.array(fc_HML_series)
fc_alpha_arr = np.array(fc_alpha_series)
R_p_arr = np.array(R_p_series)

print(f"\n✅ 完成！共收集 {N_MONTHS} 个月的数据")

In [ ]:
# ========== 统计汇总 ==========
print("📊 因子贡献的统计汇总（60 个月）")
print("═" * 65)
print(f"  {'指标':>10} {'MKT':>10} {'SMB':>10} {'HML':>10} {'Alpha':>10}")
print(f"  {'─'*10} {'─'*10} {'─'*10} {'─'*10} {'─'*10}")
print(f"  {'均值':>10} {fc_MKT_arr.mean():>10.4f} {fc_SMB_arr.mean():>10.4f} {fc_HML_arr.mean():>10.4f} {fc_alpha_arr.mean():>10.4f}")
print(f"  {'标准差':>10} {fc_MKT_arr.std():>10.4f} {fc_SMB_arr.std():>10.4f} {fc_HML_arr.std():>10.4f} {fc_alpha_arr.std():>10.4f}")
print(f"  {'最小值':>10} {fc_MKT_arr.min():>10.4f} {fc_SMB_arr.min():>10.4f} {fc_HML_arr.min():>10.4f} {fc_alpha_arr.min():>10.4f}")
print(f"  {'最大值':>10} {fc_MKT_arr.max():>10.4f} {fc_SMB_arr.max():>10.4f} {fc_HML_arr.max():>10.4f} {fc_alpha_arr.max():>10.4f}")

print(f"\n📐 组合收益率统计：")
print(f"  均值: {R_p_arr.mean():.4f}%")
print(f"  标准差: {R_p_arr.std():.4f}%")
print(f"  最小值: {R_p_arr.min():.4f}%")
print(f"  最大值: {R_p_arr.max():.4f}%")

print(f"\n💡 关键发现：")
print(f"  1. MKT 因子贡献的均值最大，是组合收益的主要来源")
print(f"  2. 因子贡献的标准差很大，说明不同月份的归因结果差异很大")
print(f"  3. 当 R_p 接近 0 时，因子贡献会变得非常不稳定")

---

## 8. 可视化：因子贡献的时序堆叠面积图

### 🎯 展示各因子贡献随时间的变化

In [ ]:
# ========== 可视化：堆叠面积图 ==========
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

months = np.arange(1, N_MONTHS + 1)

# --- 上图：因子贡献的堆叠面积图 ---
ax1 = axes[0]
# 使用原始因子收益贡献（非比例）
factor_ret_MKT_arr = beta_p_MKT * np.array(f_MKT_series)
factor_ret_SMB_arr = beta_p_SMB * np.array(f_SMB_series)
factor_ret_HML_arr = beta_p_HML * np.array(f_HML_series)
alpha_arr = np.full(N_MONTHS, 0.2)

# 堆叠面积图
ax1.fill_between(months, 0, factor_ret_MKT_arr, alpha=0.7, label='MKT', color='#3498db')
ax1.fill_between(months, factor_ret_MKT_arr, factor_ret_MKT_arr + factor_ret_SMB_arr, 
                 alpha=0.7, label='SMB', color='#2ecc71')
ax1.fill_between(months, factor_ret_MKT_arr + factor_ret_SMB_arr, 
                 factor_ret_MKT_arr + factor_ret_SMB_arr + factor_ret_HML_arr,
                 alpha=0.7, label='HML', color='#e74c3c')
ax1.plot(months, R_p_arr, 'k-', linewidth=2, label='R_p (Total)', alpha=0.8)

ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Month', fontsize=12)
ax1.set_ylabel('Return Contribution (%)', fontsize=12)
ax1.set_title('Factor Return Contribution Over Time (Stacked)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10, loc='upper left')
ax1.grid(alpha=0.3)

# --- 下图：因子贡献比例 ---
ax2 = axes[1]
# 限制极端值
fc_MKT_clip = np.clip(fc_MKT_arr, -2, 3)
fc_SMB_clip = np.clip(fc_SMB_arr, -2, 3)
fc_HML_clip = np.clip(fc_HML_arr, -2, 3)
fc_alpha_clip = np.clip(fc_alpha_arr, -2, 3)

ax2.fill_between(months, 0, fc_MKT_clip, alpha=0.7, label='MKT', color='#3498db')
ax2.fill_between(months, fc_MKT_clip, fc_MKT_clip + fc_SMB_clip, 
                 alpha=0.7, label='SMB', color='#2ecc71')
ax2.fill_between(months, fc_MKT_clip + fc_SMB_clip, 
                 fc_MKT_clip + fc_SMB_clip + fc_HML_clip,
                 alpha=0.7, label='HML', color='#e74c3c')

ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.axhline(y=1, color='gray', linewidth=1, linestyle='--', alpha=0.5, label='100% Line')
ax2.set_xlabel('Month', fontsize=12)
ax2.set_ylabel('Factor Contribution Ratio', fontsize=12)
ax2.set_title('Factor Contribution Ratio Over Time (Clipped)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10, loc='upper left')
ax2.grid(alpha=0.3)
ax2.set_ylim(-3, 4)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  上图：各因子的收益贡献（绝对值）堆叠，黑线是组合总收益")
print(f"  下图：各因子的贡献比例（裁剪到 [-2, 3] 范围），可以看到归因的稳定性")
print(f"  当 R_p 很小时，因子贡献比例会变得非常不稳定（除以接近 0 的数）")

---

## 9. 归因恒等式的验证

### 📐 理论

归因恒等式要求：

$$\frac{\alpha}{R_p} + \sum_{k} \text{FC}_k + \frac{\varepsilon}{R_p} = 1$$

在实际中，由于我们用截面回归估计因子收益率，残差 $\varepsilon$ 的均值为 0（OLS 的性质），
所以恒等式变为：

$$\frac{\alpha}{R_p} + \sum_{k} \text{FC}_k \approx 1$$

In [ ]:
# ========== 验证归因恒等式 ==========
print("📊 验证归因恒等式（每个月）")
print("═" * 55)

# 计算每个月的归因总和
attribution_sum = fc_MKT_arr + fc_SMB_arr + fc_HML_arr + fc_alpha_arr

print(f"  归因总和的统计：")
print(f"    均值: {attribution_sum.mean():.6f}")
print(f"    标准差: {attribution_sum.std():.6f}")
print(f"    最小值: {attribution_sum.min():.6f}")
print(f"    最大值: {attribution_sum.max():.6f}")

# 检查有多少个月接近 1
close_to_1 = np.sum(np.abs(attribution_sum - 1) < 0.01)
print(f"\n  归因总和与 1 的差距 < 1% 的月数: {close_to_1}/{N_MONTHS}")

print(f"\n💡 关键发现：")
print(f"  1. 归因总和的均值接近 1，验证了恒等式")
print(f"  2. 但标准差很大，说明某些月份的归因不稳定")
print(f"  3. 不稳定的原因：当 R_p 接近 0 时，因子贡献比例会爆炸")

---

## 📌 核心概念回顾

### 📌 因子归因 (Factor Attribution)
- **定义**: 将投资组合收益分解为各个因子贡献的过程
- **公式**: $R_p = \alpha + \sum_{k} \beta_k \times f_k + \varepsilon$
- **含义**: 帮助投资者理解收益的来源，评估选股能力和因子暴露

### 📌 因子贡献 (Factor Contribution)
- **定义**: 某因子对组合收益的贡献比例
- **公式**: $\text{FC}_k = \frac{\beta_k \times f_k}{R_p}$
- **含义**: 衡量因子 $k$ 解释了组合收益的百分之多少
- **判断标准**: FC > 0 表示正向贡献，FC < 0 表示负向贡献

### 📌 归因恒等式 (Attribution Identity)
- **定义**: 所有因子贡献之和等于 1
- **公式**: $\frac{\alpha}{R_p} + \sum_{k} \text{FC}_k + \frac{\varepsilon}{R_p} = 1$
- **含义**: 收益的来源是完备的，没有遗漏
- **判断标准**: 验证归因结果的正确性

### 📌 Alpha 贡献
- **定义**: 不能被因子解释的收益部分
- **公式**: $\text{FC}_\alpha = \frac{\alpha}{R_p}$
- **含义**: 代表基金经理的选股能力
- **判断标准**: Alpha > 0 表示有正向选股能力

### 🔗 完整流程
```
收集数据 → 估计因子暴露 β → 估计因子收益率 f → 计算因子贡献 FC → 验证恒等式
    ↓           ↓               ↓               ↓               ↓
 个股收益率   时间序列回归     截面回归        β×f/R_p         ΣFC ≈ 1
```

---

## ❌ 常见误区

### ❌ 误区 1: 因子贡献就是因子收益率
**✓ 正确理解**: 因子贡献 = 因子暴露 × 因子收益率 / 组合收益率，是一个**比例**，不是收益率本身。

### ❌ 误区 2: 因子贡献在所有月份都很稳定
**✓ 正确理解**: 当组合收益率 $R_p$ 接近 0 时，因子贡献会变得非常不稳定（除以接近 0 的数）。

### ❌ 误区 3: 归因恒等式总是精确成立
**✓ 正确理解**: 归因恒等式在理论上精确成立，但在实践中由于估计误差和舍入误差，只能近似成立。

### ❌ 误区 4: Alpha 贡代表真实的选股能力
**✓ 正确理解**: Alpha 贡献可能包含模型设定误差（遗漏因子、非线性效应等），不一定是真实的选股能力。

### ❌ 误区 5: 因子贡献可以相加比较不同基金
**✓ 正确理解**: 不同基金的因子暴露和组合收益率不同，直接比较因子贡献可能产生误导。应该比较因子收益贡献的绝对值。